In [1]:
import pandas as pd
import os
from selenium import webdriver
from selenium_stealth import stealth
from selenium.webdriver.common.by import By
import random
from selenium.webdriver.common.keys import Keys
from time import sleep
import time
from bs4 import BeautifulSoup
import json
import numpy as np

In [2]:
def parse_json(data):
    # Extract top-level information
    try:
        len_name_changes = len(data["data"].get("metadata", {}).get("nameChanges", []))
    except TypeError as e:
        len_name_changes = np.nan

    parsed_data = {
        "platformUserId": data["data"].get("platformInfo", {}).get("platformUserId", np.nan),
        "isVerified": data["data"].get("userInfo", {}).get("isVerified", np.nan),
        "pageviews": data["data"].get("userInfo", {}).get("pageviews", np.nan),
        "xpTier": data["data"].get("userInfo", {}).get("xpTier", np.nan),
        "isSuspicious": data["data"].get("userInfo", {}).get("isSuspicious", np.nan),
        "nameChanges": len_name_changes,
        "currentSeason": data["data"].get("metadata", {}).get("currentSeason", np.nan),
        "clearanceLevel": data["data"].get("metadata", {}).get("clearanceLevel", np.nan),
        "isOverwolfAppUser": data["data"].get("metadata", {}).get("isOverwolfAppUser", np.nan),
        "battlepassLevel": data["data"].get("metadata", {}).get("battlepassLevel", np.nan),
    }
    
    # Initialize dictionary for storing season data
    num_seasons = 37  # Considering 37 seasons
    season_data = {}
    
    for season in range(1, num_seasons + 1):
        for gamemode in ["Quick Match", "Ranked", "Standard", "Event", "Arcade"]:  # You can extend with more gamemodes
            prefix = f"season_{season}_{gamemode.lower().replace(' ', '')}_"
            season_data.update({
                prefix + "matchesplayed": np.nan,
                prefix + "matcheswon": np.nan,
                prefix + "matcheslost": np.nan,
                prefix + "kills": np.nan,
                prefix + "deaths": np.nan,
                prefix + "kdRatio": np.nan,
                prefix + "mmr": np.nan,
                prefix + "maxMmr": np.nan,
            })
    
    # Parse segments
    for segment in data["data"].get("segments", []):
        if segment.get("type") == "season":
            season_number = segment.get("attributes", {}).get("season")
            gamemode = segment.get("metadata", {}).get("gamemodeName", "Unknown")
            
            if season_number is not None and season_number <= num_seasons:
                prefix = f"season_{season_number}_{gamemode.lower().replace(' ', '')}_"
                stats = segment.get("stats", {})
                
                season_data[prefix + "matchesplayed"] = stats.get("matchesPlayed", {}).get("value", np.nan)
                season_data[prefix + "matcheswon"] = stats.get("matchesWon", {}).get("value", np.nan)
                season_data[prefix + "matcheslost"] = stats.get("matchesLost", {}).get("value", np.nan)
                season_data[prefix + "kills"] = stats.get("kills", {}).get("value", np.nan)
                season_data[prefix + "deaths"] = stats.get("deaths", {}).get("value", np.nan)
                season_data[prefix + "kdRatio"] = stats.get("kdRatio", {}).get("value", np.nan)
                season_data[prefix + "mmr"] = stats.get("mmr", {}).get("value", np.nan)
                season_data[prefix + "maxMmr"] = stats.get("maxMmr", {}).get("value", np.nan)
    
    # Parse overview section
    overview_data = {}
    for segment in data["data"].get("segments", []):
        if segment.get("type") == "overview":
            stats = segment.get("stats", {})
            for key, value in stats.items():
                feature_name = f"overview_{key.lower()}"
                overview_data[feature_name] = value.get("value", np.nan)
    
    # Parse gamemode section
    gamemode_data = {}
    for segment in data["data"].get("segments", []):
        if segment.get("type") == "gamemode":
            gamemode_name = segment.get("metadata", {}).get("gamemodeName", "Unknown").lower().replace(" ", "")
            stats = segment.get("stats", {})
            for key, value in stats.items():
                feature_name = f"gamemode_{gamemode_name}_{key.lower()}"
                gamemode_data[feature_name] = value.get("value", np.nan)
    
    # Merge all extracted data
    parsed_data.update(season_data)
    parsed_data.update(overview_data)
    parsed_data.update(gamemode_data)
    
    return parsed_data

In [69]:
# Create an instance of ChromeOptions
options = webdriver.ChromeOptions()

# User-agent rotation
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
]

user_agent = random.choice(user_agents)
options.add_argument(f"user-agent={user_agent}")

# Initialize the WebDriver with options
driver = webdriver.Chrome(options=options)

name = 'CheeseHub..'

# Apply stealth settings to the driver (assuming you have a stealth function)
stealth(
    driver,
    languages=["en-US", "en"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)


driver.get(f"https://api.tracker.gg/api/v2/r6siege/standard/profile/ubi/{name}")
    
# Wait for the page to load completely
sleep(5)

# Wait for the page to load completely (if necessary)
driver.implicitly_wait(10)  # Waits up to 10 seconds for elements to appear

# Retrieve the page source
json_content = driver.page_source

# Close the browser
driver.quit()

In [70]:
# Parse HTML structure and extract JSON inside <pre> tag
soup = BeautifulSoup(json_content, "html.parser")
pre_tag = soup.find("pre")
if not pre_tag:
    print(f"No JSON data found")


json_data = json.loads(pre_tag.text)  # Load JSON

In [71]:
import numpy as np

In [72]:
data_list=[]

parsed_json = parse_json(json_data)
data_list.append(parsed_json)

test_data = pd.DataFrame(data_list)

In [73]:
test_data

,platformUserId,isVerified,pageviews,xpTier,isSuspicious,nameChanges,currentSeason,clearanceLevel,isOverwolfAppUser,battlepassLevel,...,gamemode_standard+quickmatch_matchesplayed,gamemode_standard+quickmatch_matcheswon,gamemode_standard+quickmatch_matcheslost,gamemode_standard+quickmatch_matchesabandoned,gamemode_standard+quickmatch_timeplayed,gamemode_standard+quickmatch_kills,gamemode_standard+quickmatch_deaths,gamemode_standard+quickmatch_kdratio,gamemode_standard+quickmatch_killspermatch,gamemode_standard+quickmatch_winpercentage
0,1b41b98f-b53e-4804-85f8-d52dacda4590,False,222,None,None,0,37,122,False,20,...,39,12,0,27,0,114,31,3.68,2.92,30.77


In [74]:
# Store feature names before adding missing indicators (for alignment)
original_features = test_data.columns.tolist()

# Add missing indicators
for col in original_features:
    test_data[col + "_missing"] = test_data[col].isna().astype(int)  # 1 if missing, 0 if present

# Identify boolean columns
bool_cols = test_data.select_dtypes(include=['bool']).columns

# Convert them to integers (True → 1, False → 0)
test_data[bool_cols] = test_data[bool_cols].astype(int)



C:\Users\Ananthu.surendran\AppData\Local\Temp\ipykernel_47316\1701879083.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_data[col + "_missing"] = test_data[col].isna().astype(int)  # 1 if missing, 0 if present
C:\Users\Ananthu.surendran\AppData\Local\Temp\ipykernel_47316\1701879083.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_data[col + "_missing"] = test_data[col].isna().astype(int)  # 1 if missing, 0 if present
C:\Users\Ananthu.surendran\AppData\Local\Temp\ipykernel_47316\1701879083.py:6: PerformanceWarnin

In [75]:
test_data.drop('platformUserId', inplace=True, axis=1)

In [76]:
test_data

,isVerified,pageviews,xpTier,isSuspicious,nameChanges,currentSeason,clearanceLevel,isOverwolfAppUser,battlepassLevel,season_1_quickmatch_matchesplayed,...,gamemode_standard+quickmatch_matchesplayed_missing,gamemode_standard+quickmatch_matcheswon_missing,gamemode_standard+quickmatch_matcheslost_missing,gamemode_standard+quickmatch_matchesabandoned_missing,gamemode_standard+quickmatch_timeplayed_missing,gamemode_standard+quickmatch_kills_missing,gamemode_standard+quickmatch_deaths_missing,gamemode_standard+quickmatch_kdratio_missing,gamemode_standard+quickmatch_killspermatch_missing,gamemode_standard+quickmatch_winpercentage_missing
0,0,222,None,None,0,37,122,0,20,NaN,...,0,0,0,0,0,0,0,0,0,0


In [77]:
import joblib

In [78]:
feature_names = joblib.load("feature_names.pkl")
# Ensure all expected features exist
for col in feature_names:
    if col.endswith("_missing"):  # Missing indicators
        base_col = col.replace("_missing", "")
        test_data[col] = test_data[base_col].isna().astype(int) if base_col in test_data else 1
    else:  # Regular features
        test_data[col] = test_data.get(col, np.nan)

In [79]:
# Align with training features:
# 1. Add missing columns from training data (fill with NaN)
for col in feature_names:
    if col not in test_data.columns:
        test_data[col] = np.nan  # Preserve NaN for missing values

In [80]:
test_data

,isVerified,pageviews,xpTier,isSuspicious,nameChanges,currentSeason,clearanceLevel,isOverwolfAppUser,battlepassLevel,season_1_quickmatch_matchesplayed,...,gamemode_standard+quickmatch_matchesplayed_missing,gamemode_standard+quickmatch_matcheswon_missing,gamemode_standard+quickmatch_matcheslost_missing,gamemode_standard+quickmatch_matchesabandoned_missing,gamemode_standard+quickmatch_timeplayed_missing,gamemode_standard+quickmatch_kills_missing,gamemode_standard+quickmatch_deaths_missing,gamemode_standard+quickmatch_kdratio_missing,gamemode_standard+quickmatch_killspermatch_missing,gamemode_standard+quickmatch_winpercentage_missing
0,0,222,None,None,0,37,122,0,20,NaN,...,0,0,0,0,0,0,0,0,0,0


In [81]:
# 2. Remove extra columns not seen during training
new_data = test_data[feature_names]

In [82]:
new_data

,overview_matchesplayed,overview_matcheswon,overview_matcheslost,overview_matchesabandoned,overview_timeplayed,overview_kills,overview_deaths,overview_attackerroundswon,overview_attackerteamkillsinobj,overview_attackerenemykillsinobj,...,overview_playstyledefenderentrydenier_missing,overview_playstyledefenderintelprovider_missing,overview_playstyledefendersupporter_missing,overview_playstyledefendertrapper_missing,overview_playstyledefenderutilitydenier_missing,overview_kdratio_missing,overview_killspermatch_missing,overview_killspermin_missing,overview_winpercentage_missing,overview_timealive_missing
0,112,70,13,29,310130,707,303,0,0,36,...,0,0,0,0,0,0,0,0,0,0


In [83]:
for col in new_data.columns:
    print(col, ' : ', new_data[col])

overview_matchesplayed  :  0    112
Name: overview_matchesplayed, dtype: int64
overview_matcheswon  :  0    70
Name: overview_matcheswon, dtype: int64
overview_matcheslost  :  0    13
Name: overview_matcheslost, dtype: int64
overview_matchesabandoned  :  0    29
Name: overview_matchesabandoned, dtype: int64
overview_timeplayed  :  0    310130
Name: overview_timeplayed, dtype: int64
overview_kills  :  0    707
Name: overview_kills, dtype: int64
overview_deaths  :  0    303
Name: overview_deaths, dtype: int64
overview_attackerroundswon  :  0    0
Name: overview_attackerroundswon, dtype: int64
overview_attackerteamkillsinobj  :  0    0
Name: overview_attackerteamkillsinobj, dtype: int64
overview_attackerenemykillsinobj  :  0    36
Name: overview_attackerenemykillsinobj, dtype: int64
overview_attackerteamkillsoutobj  :  0    2
Name: overview_attackerteamkillsoutobj, dtype: int64
overview_attackerenemykillsoutobj  :  0    175
Name: overview_attackerenemykillsoutobj, dtype: int64
overview_de

In [84]:
import xgboost as xgb

# Load XGBoost model from file
loaded_model = xgb.XGBClassifier()
loaded_model.load_model("xgb_100.json")  # Load JSON model

In [85]:
# Make predictions
predictions = loaded_model.predict(new_data)


In [86]:
predictions

array([0])

In [87]:
probabilities = loaded_model.predict_proba(new_data)

In [88]:
probabilities

array([[0.9989762 , 0.00102382]], dtype=float32)

In [89]:
np.argmax(probabilities)

0

In [90]:
probabilities[0][1]*100

0.10238224640488625

In [91]:
# Load the scaler from file
loaded_scaler = joblib.load('scaler.pkl')

# Transform new data
X_new_transformed = loaded_scaler.transform(new_data)


In [94]:
from tensorflow.keras.models import load_model


In [99]:
nn_nodel = load_model("nn_model.h5")

In [100]:
y_pred = nn_nodel.predict(X_new_transformed)

1/1 [==============================] - 0s 52ms/step


In [101]:
y_pred

array([[0.02255716]], dtype=float32)